In [ ]:
%load_ext watermark


In [ ]:
import ast
import os
import re
import tarfile
import urllib

import pandas as pd


In [ ]:
%watermark -diwmuv -iv


In [ ]:
pd.options.display.float_format = "{:,.0f}".format


In [ ]:
summary_start = re.compile(r"^Simulation summary:", re.MULTILINE)
param_dict = re.compile(r"\{(?:[^{}]|\n)*\}", re.DOTALL)


def parse_file(path: str) -> dict:
    text = open(path, "r", encoding="utf-8", errors="ignore").read()
    # a) find summary block
    m = summary_start.search(text)
    if not m:
        return None
    # read lines after “Simulation summary:” until a blank line
    lines = text[m.end() :].splitlines()
    summary_lines = []
    for ln in lines[1:]:
        if not ln.strip():
            break
        summary_lines.append(ln)

    # b) find param dict
    d_match = param_dict.search(text)
    if not d_match:
        return None

    # parse metrics
    metrics = {}
    for ln in summary_lines:
        parts = ln.strip().split()
        # first token is the number (with commas), rest is the metric name
        val = int(parts[0].replace(",", ""))
        key = "_".join(parts[1:])
        metrics[key] = val

    # parse params dict
    params = ast.literal_eval(d_match.group(0))

    # combine
    return {**params, **metrics}


In [ ]:
def make_df(slug: str) -> pd.DataFrame:
    # 1. Download the tar.gz
    url = f"https://osf.io/{slug}/download"
    archive_path = f"{slug}.tar.gz"
    urllib.request.urlretrieve(url, archive_path)

    # 2. Extract into ./data/
    extract_dir = slug
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(archive_path, mode="r:gz") as tar:
        tar.extractall(extract_dir)

    # 3. Walk data folder, parse all files
    records = []
    for root, _, files in os.walk(slug):
        for fn in files:
            full = os.path.join(root, fn)
            rec = parse_file(full)
            if rec:
                records.append(rec)

    # 4. Build DataFrame
    return pd.DataFrame(records)


## Vanilla


In [ ]:
slug = "tmg6b"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame("mean")


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame("std")


## Vanilla --- big


In [ ]:
slug = "czt4b"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame("mean")


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame("std")


## UK


In [ ]:
slug = "ej5bz"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame("mean")


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame("std")


## UK --- big


In [ ]:
slug = "nm6wz"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame("mean")


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame("std")


## multistrain


In [ ]:
slug = "qdwb7"
df = make_df(slug)
df.to_csv(f"{slug}.csv", index=False)


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].mean(
    numeric_only=True
).to_frame("mean")


In [ ]:
df.loc[:, df.columns.str.startswith("cumulative_")].std(
    numeric_only=True
).to_frame("std")
